# Stage LR — Generation Backbone Comparison (expanded, round 2)

**Exploratory Stage LR research only — does not touch `main`, per the architecture freeze (`VALIDATION.md` §56).** This notebook does not train anything. It reuses the project's own **already-validated** phoneme-constrained decoding mechanism (`rephrase.py::PhonemeConstraintLogitsProcessor`) and asks: **does a different/larger open generation model produce better candidates than the currently-deployed one, and if so, is any of them actually good enough to build further on?**

**Round 1 result (12 sentences, `DECISION_LOG.md` 2026-09-08-A), for context:**

| Backbone | CLEAN rate |
|---|---|
| `google/flan-t5-base` | 8.3% (perfect leak-safety, but mostly via deleting the clause with the blocked word) |
| `Vamsi/T5_Paraphrase_Paws` (current production default) | 36.4% |
| `Qwen/Qwen2.5-3B-Instruct` | 54.5% |

One data point, not a conclusion — this round exists specifically to check whether it holds up at real scale, per direct instruction, rather than be treated as settled.

**What this round changes:**
1. **~80 sentences** instead of 12 — the full 36-run fresh corpus (`eval/step3_gencheck_corpus.py`, the same material behind the 21.4% figure) plus a stratified sample from the larger, well-studied R10 corpus (`eval/r10_corpus.json`, 398 runs).
2. **4 open, ungated models across a size range**, not just one: `Qwen2.5-1.5B-Instruct` (smaller/faster), `Qwen2.5-3B-Instruct` (round 1's model, kept as the reference point), `microsoft/Phi-3.5-mini-instruct` (~3.8B, a different model family at a similar size class), `Qwen2.5-7B-Instruct` (bigger, loaded 4-bit quantized to fit a free-tier GPU). All Apache-2.0/MIT-class licensed, no Hub authentication needed, verified as loadable rather than assumed from reputation, per this project's own standing practice (R23).
3. **Two known rough edges from round 1, fixed**: prompts are now stripped by removing the decoded prompt STRING rather than slicing by token count (round 1's merged-word glitch, "tointerest"/"totalinterest", was a token-boundary artifact from index-based slicing), and stray chat-template special tokens (round 1's `<tool_call>` leak) are stripped post-generation.
4. **A stated "genuinely usable" bar, not just "beats the current default"**: proposed at ≥70% CLEAN with zero SEVERE `FACTUAL_OR_LOGICAL_REVERSAL` defects — a wrong-but-fluent factual reversal is worse than an awkward-but-honest sentence, per this project's own prior findings on that defect class. Applied when judging results back in the main session, not inside this notebook.

**Honest disclosure:** written and reviewed carefully, not executed — no GPU/Colab access in the environment that wrote it. The 7B model in particular may need a runtime restart or a smaller model dropped if it doesn't fit in your specific GPU allocation; the notebook is written to skip a model that fails to load rather than abort the whole run, and says so plainly if it happens.

**Quality judging happens back in the main session, not here** — same discipline as round 1: raw outputs and hard mechanical checks only, blind Claude judging afterward with no backbone identity revealed.

## 0. Setup — GPU runtime required

**Runtime > Change runtime type > T4 GPU (or better)** before running, or the larger model in section 3 will not fit / will be impractically slow.

In [ ]:
# IMPORTANT: torch/torchvision are deliberately NOT upgraded here. Colab
# ships a matched, working torch+torchvision pair; upgrading torch alone
# (a mistake in an earlier version of this notebook) desyncs torchvision's
# compiled ops from it, and transformers' auto-loading machinery then fails
# with a confusing "ModuleNotFoundError: Could not import module
# 'Qwen2ForCausalLM'" that actually masks a "torchvision::nms does not
# exist" RuntimeError underneath. Only pure-Python libraries are touched.
#
# wordfreq is required transitively: rephrase.py -> semantic.py -> freq.py
# -> wordfreq.zipf_frequency (project's own memory-safe wrapper). Missing
# it breaks the `import rephrase` cell below with an unrelated-looking
# ModuleNotFoundError.
#
# bitsandbytes added this round: Qwen2.5-7B-Instruct is loaded 4-bit
# quantized to fit a free-tier (T4, 16GB) GPU alongside the other models.
!pip install -q -U transformers accelerate sentencepiece nltk wordfreq bitsandbytes

In [ ]:
# Sanity check torch/torchvision are actually paired correctly BEFORE
# anything else runs, so a mismatch surfaces here clearly instead of deep
# inside a Qwen2ForCausalLM import failure later.
import torch
print("torch:", torch.__version__)
try:
    import torchvision
    print("torchvision:", torchvision.__version__)
    _ = torch.ops.torchvision.nms  # the exact op that broke before -- probe it directly
    print("torch/torchvision compiled ops OK.")
except Exception as e:
    raise RuntimeError(
        "torch/torchvision mismatch detected (this is the failure mode that broke "
        "an earlier run of this notebook). Do NOT try to fix by upgrading torch "
        "alone. Instead: Runtime > Disconnect and delete runtime, then Runtime > "
        "Run all on a fresh instance -- Colab's preinstalled pair should be "
        "consistent on a clean VM."
    ) from e

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Pull the already-validated pieces directly from the repo

No participant data involved here — `rephrase.py`, `phonetic.py`, and `eval/step3_gencheck_corpus.py` (the fresh generalization-check corpus, `VALIDATION.md` §54) are all tracked, public, non-sensitive files.

In [ ]:
!rm -rf speech-ai
!git clone --branch stage-lr --depth 1 https://github.com/haqiqak/speech-ai.git
import sys
sys.path.insert(0, "speech-ai")

In [ ]:
import nltk
# wordnet/omw-1.4 added defensively: semantic.py (imported transitively by
# rephrase.py) does `from nltk.corpus import wordnet as wn` at module level
# and instantiates a WordNetLemmatizer -- cheap to pre-download rather than
# risk a lazy-load failure mid-run.
for pkg in ["cmudict", "wordnet", "omw-1.4", "punkt", "punkt_tab",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"skip {pkg}: {e}")

In [ ]:
import os
os.environ["REPHRASE_DEVICE"] = "cuda"

import importlib
import phonetic
import rephrase
from rephrase import PhonemeConstraintLogitsProcessor

print("imported rephrase.py / phonetic.py from the tracked repo directly -- not reimplemented.")

## 2. Eval material — ~80 sentences from two tracked, non-sensitive corpora

The full `eval/step3_gencheck_corpus.py` (36 runs, the same material behind the 21.4% fresh-corpus figure) plus a stratified sample from `eval/r10_corpus.json`/`r10_run_plan.json` (398 runs, the larger corpus behind the 31-34% plateau) — more volume, and a second, independent corpus so a result isn't an artifact of one specific text sample.

In [ ]:
sys.path.insert(0, "speech-ai/eval")
from step3_gencheck_corpus import CORPUS, RUN_PLAN

by_id = {c["id"]: c for c in CORPUS}

# Use the FULL step3 corpus this round (36 runs), not a subset -- round 1
# already showed the mechanism handles 12 fine; the open question now is
# quality at scale.
def step3_sentence_and_blocked(run):
    sent = by_id[run["id"]]["text"]
    return {"corpus": "step3", "id": run["id"], "text": sent,
            "words": run.get("words", []), "sounds": run.get("sounds", [])}

STEP3_ITEMS = [step3_sentence_and_blocked(r) for r in RUN_PLAN]
print(f"step3: {len(STEP3_ITEMS)} items")

# R10 corpus: words/sounds live nested under run["spec"], unlike step3's
# flat structure -- different corpus, different schema, handled explicitly
# rather than assumed to match.
import json as _json
r10_corpus = _json.loads(open("speech-ai/eval/r10_corpus.json", encoding="utf-8").read())
r10_run_plan = _json.loads(open("speech-ai/eval/r10_run_plan.json", encoding="utf-8").read())["runs"]
r10_by_id = {r["sentence_id"]: r["sentence_text"] for r in r10_corpus["records"]}

def r10_sentence_and_blocked(run):
    sent = r10_by_id.get(run["sentence_id"])
    spec = run.get("spec", {})
    return {"corpus": "r10", "id": run["profile_id"], "text": sent,
            "words": spec.get("words", []), "sounds": spec.get("sounds", [])}

# Stratified sample: every 8th run, skipping any whose sentence text wasn't
# found (a few R10 sentence_ids appear in run_plan but not in every corpus
# export -- skip rather than crash) and any with zero declared words/sounds
# (nothing to avoid, not a useful test case for this comparison).
r10_candidates = [r10_sentence_and_blocked(r) for r in r10_run_plan[::8]]
R10_ITEMS = [r for r in r10_candidates if r["text"] and (r["words"] or r["sounds"])]
print(f"r10 (stratified sample): {len(R10_ITEMS)} items")

ALL_ITEMS = STEP3_ITEMS + R10_ITEMS
print(f"total: {len(ALL_ITEMS)} items")

def sentence_and_blocked(item):
    return item["text"], item["words"], item["sounds"]

## 3. Backbones to compare

1. **Current production default** — `Vamsi/T5_Paraphrase_Paws` (control).
2. **`google/flan-t5-base`** — round 1's mechanically-safest-but-quality-poor result; kept in to see if the pattern (safety via deletion) holds at scale.
3. **`Qwen/Qwen2.5-1.5B-Instruct`** — smaller/faster than round 1's model; checks whether a smaller size in the same family still clears the bar, at lower compute cost if so.
4. **`Qwen/Qwen2.5-3B-Instruct`** — round 1's model, kept as the reference point for this round's comparison.
5. **`microsoft/Phi-3.5-mini-instruct`** — a different model family at a similar size class (~3.8B) to Qwen2.5-3B, checking whether round 1's result is about "this model" or "models at this scale/training approach" more generally.
6. **`Qwen/Qwen2.5-7B-Instruct`** — bigger, loaded 4-bit quantized to fit a free-tier GPU; checks whether more scale keeps helping or plateaus.

In [ ]:
SEQ2SEQ_BACKBONES = ["Vamsi/T5_Paraphrase_Paws", "google/flan-t5-base"]  # bare "flan-t5-base" 404s -- needs the org prefix
CAUSAL_BACKBONES = [
    ("Qwen/Qwen2.5-1.5B-Instruct", False),   # (model_name, load_4bit)
    ("Qwen/Qwen2.5-3B-Instruct", False),
    ("microsoft/Phi-3.5-mini-instruct", False),
    ("Qwen/Qwen2.5-7B-Instruct", True),      # 4-bit: ~14GB fp16 weights alone won't fit safely alongside everything else on a T4
]

In [ ]:
import re as _re

# Chat-template artifacts seen in round 1 (a stray "<tool_call>" token) --
# stripped defensively regardless of which model produces them.
_SPECIAL_TOKEN_ARTIFACT_RE = _re.compile(r"<\|?/?(tool_call|im_start|im_end)\|?>", _re.IGNORECASE)


def run_seq2seq_backbone(model_name, items, k=5):
    """Reuses generate_candidates_phoneme_constrained() UNMODIFIED --
    only the module-level model pointer is swapped between calls."""
    os.environ["REPHRASE_MODEL"] = model_name
    rephrase._model = None
    rephrase._tokenizer = None
    rephrase._load_tried = False
    rephrase._rephrase_ok = False
    rephrase.REPHRASE_MODEL = model_name

    results = []
    for item in items:
        sent, blocked_words, blocked_patterns = sentence_and_blocked(item)
        import time
        t0 = time.perf_counter()
        candidates, stats = rephrase.generate_candidates_phoneme_constrained(
            sent, k=k, blocked_words=blocked_words, blocked_patterns=blocked_patterns,
        )
        latency = time.perf_counter() - t0
        results.append({
            "run_id": item["id"], "corpus": item["corpus"], "original": sent,
            "blocked_words": blocked_words, "blocked_patterns": blocked_patterns,
            "candidates": candidates, "stats": stats, "latency_s": round(latency, 3),
        })
    return results


def _load_causal_tokenizer(model_name):
    """Some model repos (Phi family, historically) need trust_remote_code
    for their tokenizer; current transformers has native support for the
    models used here, so this is a fallback, not an assumption either way."""
    from transformers import AutoTokenizer
    try:
        return AutoTokenizer.from_pretrained(model_name)
    except ValueError as e:
        if "trust_remote_code" not in str(e):
            raise
        return AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


def _load_causal_model(model_name, load_4bit):
    """dtype= replaced torch_dtype= in newer transformers releases; try the
    current name first and fall back for older pinned versions. 4-bit
    quantization (bitsandbytes) is opt-in per model, for the one (7B) that
    won't reliably fit a free-tier GPU in full fp16 alongside everything
    else loaded/unloaded across this run. trust_remote_code follows the
    same try-then-fallback pattern as the tokenizer, for the same reason."""
    from transformers import AutoModelForCausalLM
    kwargs = {"device_map": "cuda"}
    if load_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        kwargs["dtype"] = torch.float16

    def _try(load_kwargs):
        try:
            return AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        except TypeError:
            if "dtype" in load_kwargs:
                load_kwargs = dict(load_kwargs)
                load_kwargs["torch_dtype"] = load_kwargs.pop("dtype")
                return AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
            raise

    try:
        return _try(kwargs)
    except ValueError as e:
        if "trust_remote_code" not in str(e):
            raise
        return _try({**kwargs, "trust_remote_code": True})


def _extract_reply(tok, seq, prompt_text):
    """Round 1's merged-word glitch ("tointerest", "totalinterest") was
    traced to decoding seq[prompt_len:] by TOKEN INDEX -- when a token
    straddles the prompt/completion boundary, slicing by index can drop
    the space-marking part of that token, silently gluing two words
    together. Fixed by decoding the FULL sequence to text first, then
    removing the decoded PROMPT as a text prefix -- comparing decoded
    strings sidesteps the token-boundary issue entirely, since nothing is
    ever sliced mid-token. Falls back to the old index-based slice only if
    the text-prefix match fails (e.g. a tokenizer whose decode() output
    isn't a clean prefix-superset relationship), so a model can never
    silently produce zero output because of this fallback path."""
    full_text = tok.decode(seq, skip_special_tokens=True)
    if full_text.startswith(prompt_text):
        reply = full_text[len(prompt_text):]
    else:
        prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
        reply = tok.decode(seq[len(prompt_ids):], skip_special_tokens=True)
    reply = _SPECIAL_TOKEN_ARTIFACT_RE.sub("", reply).strip()
    return reply


def run_causal_backbone(model_name, items, k=5, load_4bit=False):
    """Adapts the SAME PhonemeConstraintLogitsProcessor to a causal LM's
    generate() call. decoder_start_len must be the prompt's token length
    (input_ids during causal generation includes the full prompt, not just
    the decoder's own output) -- otherwise the processor could flag a
    blocked word that legitimately appears in the prompt itself.

    Returns None (and prints why) instead of raising if the model can't be
    loaded (OOM, etc.) -- per this notebook's own stated design, one model
    failing to load should not abort the whole run."""
    from transformers import LogitsProcessorList
    try:
        tok = _load_causal_tokenizer(model_name)
        model = _load_causal_model(model_name, load_4bit)
    except Exception as e:
        print(f"  SKIPPING {model_name}: failed to load ({type(e).__name__}: {e})")
        return None

    results = []
    for item in items:
        sent, blocked_words, blocked_patterns = sentence_and_blocked(item)
        avoid_note = ""
        if blocked_words:
            avoid_note += f" Do not use these words: {', '.join(blocked_words)}."
        messages = [{
            "role": "user",
            "content": f"Reword this sentence so it means exactly the same thing, in natural English. "
                       f"Output only the reworded sentence, nothing else.{avoid_note}\n\nSentence: {sent}",
        }]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_text_decoded = tok.decode(tok(prompt, return_tensors="pt")["input_ids"][0], skip_special_tokens=True)
        encoded = tok(prompt, return_tensors="pt").to("cuda")
        prompt_len = encoded["input_ids"].shape[1]
        processor = PhonemeConstraintLogitsProcessor(tok, blocked_patterns, decoder_start_len=prompt_len)

        import time
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **encoded, max_new_tokens=80, num_beams=4, num_return_sequences=min(k, 4),
                no_repeat_ngram_size=3, early_stopping=True,
                logits_processor=LogitsProcessorList([processor]),
                pad_token_id=tok.eos_token_id,
            )
        latency = time.perf_counter() - t0
        candidates = []
        for seq in out:
            reply = _extract_reply(tok, seq, prompt_text_decoded)
            if reply and reply not in candidates:
                candidates.append(reply)
        results.append({
            "run_id": item["id"], "corpus": item["corpus"], "original": sent,
            "blocked_words": blocked_words, "blocked_patterns": blocked_patterns,
            "candidates": candidates[:k], "stats": {"beam_kills": processor.kill_count},
            "latency_s": round(latency, 3),
        })
    del model
    torch.cuda.empty_cache()
    return results

In [ ]:
all_results = {}
for backbone in SEQ2SEQ_BACKBONES:
    print(f"=== {backbone} ===")
    all_results[backbone] = run_seq2seq_backbone(backbone, ALL_ITEMS)
    print(f"  done, {len(all_results[backbone])} runs")

for backbone, load_4bit in CAUSAL_BACKBONES:
    print(f"=== {backbone}{' (4-bit)' if load_4bit else ''} ===")
    result = run_causal_backbone(backbone, ALL_ITEMS, load_4bit=load_4bit)
    if result is None:
        print(f"  {backbone} skipped -- see the message above for why (likely OOM). "
              f"Continuing with the remaining backbones.")
        continue
    all_results[backbone] = result
    print(f"  done, {len(result)} runs")

print(f"\nBackbones that completed: {list(all_results.keys())}")

## 4. Hard mechanical checks — not judgment calls

Verify directly, don't trust the processor's own claim: does any surviving candidate actually still contain a blocked sound/word? This is checkable by code, same `phonetic.matches_any()` the live pipeline itself uses.

In [ ]:
import json
import re

def verify_leak_free(results):
    """Two different, both-useful counts, not conflated into one:
    - runs_with_any_leaking_candidate: at least one of the k candidates
      still contains a blocked word/pattern (informative but not fatal --
      the pipeline only needs ONE clean candidate per run).
    - runs_with_no_clean_candidate: none of the k candidates are clean --
      this is the one that actually matters for real-world usability.
    (An earlier version of this function incremented a single "n_leaked"
    counter once per LEAKING CANDIDATE while labeling it
    "runs_with_a_leak" -- a real bug, found by manually re-deriving the
    numbers against a real run's output. Fixed here to count per-run,
    both ways, explicitly.)"""
    n_runs = len(results)
    n_with_candidates = 0
    n_runs_with_any_leak = 0
    n_runs_with_no_clean_candidate = 0
    total_latency = 0.0
    for r in results:
        total_latency += r["latency_s"]
        if r["candidates"]:
            n_with_candidates += 1
        leaking_candidates = set()
        for cand in r["candidates"]:
            for w in re.findall(r"[A-Za-z][A-Za-z'-]*", cand):
                if any(bw.lower() == w.lower() for bw in r["blocked_words"]) or \
                   phonetic.matches_any(w, r["blocked_patterns"]):
                    leaking_candidates.add(cand)
                    break
        if leaking_candidates:
            n_runs_with_any_leak += 1
        n_clean = len(r["candidates"]) - len(leaking_candidates)
        if n_clean == 0:
            n_runs_with_no_clean_candidate += 1
    return {
        "n_runs": n_runs,
        "runs_with_any_candidate": n_with_candidates,
        "runs_with_any_leaking_candidate": n_runs_with_any_leak,
        "runs_with_no_clean_candidate": n_runs_with_no_clean_candidate,
        "mean_latency_s": round(total_latency / max(1, n_runs), 2),
    }

summary = {name: verify_leak_free(res) for name, res in all_results.items()}
print(json.dumps(summary, indent=2))
print("\nThe number that actually disqualifies a backbone is 'runs_with_no_clean_candidate' --\n"
      "a run with SOME leaking candidates but at least one clean one is still fine, since the\n"
      "real pipeline only needs to find one usable candidate per sentence.")

## 5. Save raw outputs for blind judging back in the main session

Meaning-preservation and naturalness are judgment calls, not mechanical checks — per this project's own discipline, those get judged blind (no metadata, same rubric as every prior phase) via a fresh Claude call in the main session, not scored here.

In [ ]:
import json
from pathlib import Path

output = {"mechanical_summary": summary, "raw_results": all_results}
Path("generation_backbone_comparison_results.json").write_text(json.dumps(output, indent=2))
print("saved generation_backbone_comparison_results.json")

try:
    from google.colab import files
    files.download("generation_backbone_comparison_results.json")
except ImportError:
    print("Not in Colab -- find the file in the working directory.")

## 6. What to do with this

1. Check the mechanical summary first (section 4) — a backbone with a high leak rate is disqualified regardless of how good its phrasing looks, no exception.
2. Bring `generation_backbone_comparison_results.json` back to the main session for blind quality judging (same rubric as round 1: meaning preservation, naturalness, grammaticality, plus meaning-loss and factual-reversal as explicit defect categories).
3. Judge each backbone against the stated bar for "genuinely usable," not just "better than the current default": **≥70% CLEAN, with zero SEVERE `FACTUAL_OR_LOGICAL_REVERSAL` defects.** A backbone that clears this is worth investing further in; one that doesn't (even if it beats the current default) is a data point, not a direction to commit to yet.
4. This is still Stage LR research — a backbone clearing the bar here is a candidate for further Stage LR work (e.g. LR.4-style fine-tuning once real preference data exists, per `LEARNED_REFORMULATION_RESEARCH.md`), not an authorized change to `main`'s frozen `rephrase.py`, per the freeze.